# Optimización de Estrategia de Muestreo Científico
## Modelo MILP - Círculos de Hadas

Este notebook resuelve un modelo de Programación Lineal Entera Mixta para optimizar la estrategia de muestreo respetando restricciones de presupuesto, tiempo y requisitos técnicos.

In [ ]:
# Instalar dependencias
!pip install pulp pandas numpy matplotlib seaborn openpyxl -q

In [ ]:
import pulp
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from io import StringIO
import json

print(f"PuLP version: {pulp.__version__}")

## 1. Definición del Modelo MILP

In [ ]:
def crear_modelo_optimizacion(params):
    """
    Crea y resuelve el modelo MILP de optimización de muestreo.
    
    Args:
        params: diccionario con todos los parámetros del modelo
    
    Returns:
        tuple: (modelo, status, resultados_procesados)
    """
    
    # Extraer parámetros
    circulos = params['circulos']
    separaciones = params['separaciones']
    tipos_lab = params['tipos_lab']
    
    T_total = params['T_total']
    h_dia = params['h_dia']
    t_std = params['t_std']
    t_multi = params['t_multi']
    t_24h = params['t_24h']
    t_extendida = params['t_extendida']
    
    B_total = params['B_total']
    costo_diario = params['costo_diario']
    costo_lab = params['costo_lab']
    
    min_multi_por_circulo = params['min_multi_por_circulo']
    min_24h_total = params['min_24h_total']
    min_ext_total = params['min_ext_total']
    min_lab_por_tipo = params['min_lab_por_tipo']
    E_max = params['E_max']
    
    pesos = params['pesos']
    
    # Crear modelo
    modelo = pulp.LpProblem("Optimizacion_Muestreo", pulp.LpMaximize)
    
    # --- VARIABLES DE DECISIÓN ---
    # Selección de separación por círculo
    x = {(i, s): pulp.LpVariable(f"sep_{i}_{s}", cat='Binary') 
         for i in circulos for s in separaciones}
    
    # Puntos multinivel por círculo
    m = {i: pulp.LpVariable(f"multinivel_{i}", lowBound=0, cat='Integer') 
         for i in circulos}
    
    # Equipos de trabajo
    E = pulp.LpVariable("equipos", lowBound=1, upBound=E_max, cat='Integer')
    
    # Horas-hombre totales
    H = pulp.LpVariable("horas_hombre", lowBound=0, cat='Integer')
    
    # Mediciones de larga duración
    M_24h = pulp.LpVariable("mediciones_24h", lowBound=0, cat='Integer')
    M_ext = pulp.LpVariable("mediciones_extendidas", lowBound=0, cat='Integer')
    
    # Muestras de laboratorio por tipo
    L = {k: pulp.LpVariable(f"lab_{k}", lowBound=0, cat='Integer') 
         for k in tipos_lab}
    
    # --- PARÁMETROS CALCULADOS ---
    # Número de puntos estándar por separación y círculo
    perimetros = params['perimetros']  # diccionario {circulo: perímetro}
    n_std = {}
    for i in circulos:
        for s in separaciones:
            n_std[(i, s)] = int(perimetros[i] / s) if s > 0 else 0
    
    # --- FUNCIÓN OBJETIVO ---
    # Maximizar calidad científica total
    calidad = (pulp.lpSum([pesos['std'] * n_std[(i, s)] * x[(i, s)] 
                           for i in circulos for s in separaciones]) +
              pulp.lpSum([pesos['multi'] * m[i] for i in circulos]) +
              pesos['24h'] * M_24h +
              pesos['ext'] * M_ext +
              pulp.lpSum([pesos['lab'][k] * L[k] for k in tipos_lab]))
    
    modelo += calidad, "Calidad_Cientifica"
    
    # --- RESTRICCIONES ---
    
    # R1: Definición de esfuerzo (horas totales)
    for i in circulos:
        for s in separaciones:
            modelo += (H >= (n_std[(i, s)] * t_std + m[i] * t_multi + 
                           M_24h * t_24h + M_ext * t_extendida - 
                           10000 * (1 - x[(i, s)])),
                      f"esfuerzo_{i}_{s}")
    
    # R2: Capacidad operativa (tiempo)
    dias_disponibles = T_total / h_dia
    modelo += H <= E * dias_disponibles * h_dia, "capacidad_tiempo"
    
    # R3: Presupuesto
    costo_total = (E * costo_diario * dias_disponibles +
                   pulp.lpSum([L[k] * costo_lab[k] for k in tipos_lab]))
    modelo += costo_total <= B_total, "presupuesto"
    
    # R4: Selección única de separación por círculo
    for i in circulos:
        modelo += pulp.lpSum([x[(i, s)] for s in separaciones]) == 1, f"selec_sep_{i}"
    
    # R5: Mínimos de mediciones de larga duración
    modelo += M_24h >= min_24h_total, "min_24h"
    modelo += M_ext >= min_ext_total, "min_ext"
    
    # R6: Mínimos de laboratorio
    for k in tipos_lab:
        modelo += L[k] >= min_lab_por_tipo[k], f"min_lab_{k}"
    
    # R7: Mínimo de puntos multinivel por círculo
    for i in circulos:
        modelo += m[i] >= min_multi_por_circulo, f"min_multi_{i}"
    
    return modelo, x, m, E, H, M_24h, M_ext, L, n_std, dias_disponibles

print("✓ Función de modelo MILP definida")

## 2. Parámetros de Ejemplo

In [ ]:
# PARÁMETROS DE ENTRADA
parametros_ejemplo = {
    # Geometría
    'circulos': ['HF_001', 'HF_002', 'HF_003'],
    'perimetros': {'HF_001': 1500, 'HF_002': 2000, 'HF_003': 1800},
    'separaciones': [50, 75, 100, 150],  # metros
    
    # Tiempo (horas)
    'T_total': 480,  # 2 meses de trabajo
    'h_dia': 8,
    't_std': 2,  # tiempo por punto estándar
    't_multi': 4,  # tiempo por punto multinivel
    't_24h': 24,  # medición de 24h
    't_extendida': 72,  # medición extendida
    
    # Presupuesto (USD)
    'B_total': 50000,
    'costo_diario': 200,  # salarios + manutención
    'costo_lab': {
        'mineralogico': 150,
        'biogeoquimico': 200,
        'cromatografia': 250,
        'deuterio': 300,
        'helio': 350
    },
    'tipos_lab': ['mineralogico', 'biogeoquimico', 'cromatografia', 'deuterio', 'helio'],
    
    # Mínimos requeridos
    'min_multi_por_circulo': 3,
    'min_24h_total': 5,
    'min_ext_total': 2,
    'min_lab_por_tipo': {
        'mineralogico': 5,
        'biogeoquimico': 4,
        'cromatografia': 3,
        'deuterio': 2,
        'helio': 2
    },
    'E_max': 4,  # máximo 4 equipos
    
    # Pesos de calidad científica
    'pesos': {
        'std': 1.0,
        'multi': 1.5,
        '24h': 2.0,
        'ext': 2.5,
        'lab': {
            'mineralogico': 1.2,
            'biogeoquimico': 1.5,
            'cromatografia': 1.8,
            'deuterio': 2.0,
            'helio': 2.2
        }
    }
}

print("✓ Parámetros de ejemplo cargados")
print(f"Círculos: {parametros_ejemplo['circulos']}")
print(f"Presupuesto: ${parametros_ejemplo['B_total']:,}")
print(f"Tiempo disponible: {parametros_ejemplo['T_total']} horas")

## 3. Resolver el Modelo

In [ ]:
# Crear y resolver el modelo
modelo, x, m, E, H, M_24h, M_ext, L, n_std, dias_disponibles = crear_modelo_optimizacion(parametros_ejemplo)

# Usar CBC solver (incluido en PuLP)
print("Resolviendo modelo...\n")
status = modelo.solve(pulp.PULP_CBC_CMD(msg=0))

print(f"Estado: {pulp.LpStatus[status]}")
print(f"Calidad Científica Total: {pulp.value(modelo.objective):.2f}")
print(f"\nVariables principales:")
print(f"  - Equipos de trabajo: {int(E.varValue)}")
print(f"  - Horas-hombre totales: {int(H.varValue)}")
print(f"  - Mediciones 24h: {int(M_24h.varValue)}")
print(f"  - Mediciones extendidas: {int(M_ext.varValue)}")

## 4. Procesar Resultados

In [ ]:
def procesar_resultados_modelo(modelo, parametros, x, m, E, H, M_24h, M_ext, L, n_std, dias_disponibles):
    """
    Procesa los resultados del modelo resuelto.
    """
    
    resultados = {
        'status': pulp.LpStatus[modelo.status],
        'calidad_total': float(pulp.value(modelo.objective)),
        'recursos_utilizados': {},
        'estrategia_por_circulo': {},
        'mediciones': {},
        'laboratorio': {}
    }
    
    # Recursos utilizados
    equipos = int(E.varValue)
    horas_total = int(H.varValue)
    dias_usados = horas_total / parametros['h_dia'] / equipos if equipos > 0 else 0
    
    costo_campo = equipos * parametros['costo_diario'] * dias_disponibles
    costo_lab_total = sum(L[k].varValue * parametros['costo_lab'][k] 
                         for k in parametros['tipos_lab'])
    costo_total = costo_campo + costo_lab_total
    
    resultados['recursos_utilizados'] = {
        'equipos_trabajo': equipos,
        'horas_hombre': horas_total,
        'dias_estimados': round(dias_usados, 1),
        'costo_campo': round(costo_campo, 2),
        'costo_laboratorio': round(costo_lab_total, 2),
        'costo_total': round(costo_total, 2),
        'presupuesto_disponible': parametros['B_total'],
        'presupuesto_utilizado_%': round(100 * costo_total / parametros['B_total'], 1)
    }
    
    # Estrategia por círculo
    for i in parametros['circulos']:
        for s in parametros['separaciones']:
            if x[(i, s)].varValue == 1:
                n_puntos = n_std[(i, s)]
                m_puntos = int(m[i].varValue)
                resultados['estrategia_por_circulo'][i] = {
                    'separacion_m': s,
                    'puntos_estandar': n_puntos,
                    'puntos_multinivel': m_puntos,
                    'puntos_totales': n_puntos + m_puntos
                }
    
    # Mediciones de larga duración
    resultados['mediciones'] = {
        'mediciones_24h': int(M_24h.varValue),
        'mediciones_extendidas': int(M_ext.varValue),
        'total_mediciones': int(M_24h.varValue + M_ext.varValue)
    }
    
    # Análisis de laboratorio
    for k in parametros['tipos_lab']:
        resultados['laboratorio'][k] = int(L[k].varValue)
    resultados['laboratorio']['total'] = sum(resultados['laboratorio'].values())
    
    return resultados

# Procesar resultados
resultados = procesar_resultados_modelo(modelo, parametros_ejemplo, x, m, E, H, M_24h, M_ext, L, n_std, dias_disponibles)

print("\n=== RESUMEN DE RESULTADOS ===")
print(f"\nEstado: {resultados['status']}")
print(f"Calidad Científica Total: {resultados['calidad_total']:.2f}")

print(f"\nRECURSOS UTILIZADOS:")
for key, value in resultados['recursos_utilizados'].items():
    print(f"  {key}: {value}")

print(f"\nESTRATEGIA POR CÍRCULO:")
for circulo, datos in resultados['estrategia_por_circulo'].items():
    print(f"  {circulo}: separación {datos['separacion_m']}m, {datos['puntos_totales']} puntos")

print(f"\nMEDICIONES DE LARGA DURACIÓN:")
for key, value in resultados['mediciones'].items():
    print(f"  {key}: {value}")

print(f"\nANÁLISIS DE LABORATORIO:")
for tipo, cantidad in resultados['laboratorio'].items():
    print(f"  {tipo}: {cantidad}")

## 5. Visualizaciones

In [ ]:
# Crear visualizaciones
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('Resultados de Optimización de Estrategia de Muestreo', fontsize=16, fontweight='bold')

# 1. Distribución de puntos por círculo
circulos_lista = list(resultados['estrategia_por_circulo'].keys())
puntos_std = [resultados['estrategia_por_circulo'][c]['puntos_estandar'] for c in circulos_lista]
puntos_multi = [resultados['estrategia_por_circulo'][c]['puntos_multinivel'] for c in circulos_lista]

x_pos = np.arange(len(circulos_lista))
axes[0, 0].bar(x_pos, puntos_std, label='Estándar', alpha=0.8, color='steelblue')
axes[0, 0].bar(x_pos, puntos_multi, bottom=puntos_std, label='Multinivel', alpha=0.8, color='coral')
axes[0, 0].set_xlabel('Círculo')
axes[0, 0].set_ylabel('Número de Puntos')
axes[0, 0].set_title('Puntos de Muestreo por Círculo')
axes[0, 0].set_xticks(x_pos)
axes[0, 0].set_xticklabels(circulos_lista)
axes[0, 0].legend()
axes[0, 0].grid(axis='y', alpha=0.3)

# 2. Presupuesto utilizado
categorias = ['Campo', 'Laboratorio']
costos = [resultados['recursos_utilizados']['costo_campo'],
         resultados['recursos_utilizados']['costo_laboratorio']]
colors_presupuesto = ['#2ecc71', '#e74c3c']
axes[0, 1].pie(costos, labels=categorias, autopct='%1.1f%%', colors=colors_presupuesto,
              startangle=90, textprops={'fontsize': 10})
axes[0, 1].set_title(f"Distribución de Presupuesto\nTotal: ${resultados['recursos_utilizados']['costo_total']:,.0f}")

# 3. Análisis de laboratorio
tipos = list(resultados['laboratorio'].keys())[:-1]  # Excluir 'total'
cantidades = [resultados['laboratorio'][t] for t in tipos]
axes[1, 0].barh(tipos, cantidades, color='mediumseagreen')
axes[1, 0].set_xlabel('Número de Muestras')
axes[1, 0].set_title('Análisis de Laboratorio por Tipo')
axes[1, 0].grid(axis='x', alpha=0.3)

# 4. Mediciones de larga duración
med_tipos = ['24 horas', 'Extendidas']
med_vals = [resultados['mediciones']['mediciones_24h'],
           resultados['mediciones']['mediciones_extendidas']]
colors_med = ['#3498db', '#9b59b6']
axes[1, 1].bar(med_tipos, med_vals, color=colors_med, alpha=0.8, edgecolor='black', linewidth=1.5)
axes[1, 1].set_ylabel('Número de Mediciones')
axes[1, 1].set_title('Mediciones de Larga Duración')
axes[1, 1].grid(axis='y', alpha=0.3)
for i, v in enumerate(med_vals):
    axes[1, 1].text(i, v + 0.1, str(v), ha='center', fontweight='bold')

plt.tight_layout()
plt.savefig('/tmp/resultados_optimizacion.png', dpi=150, bbox_inches='tight')
print("✓ Gráficas guardadas")
plt.show()

## 6. Exportar Resultados

In [ ]:
# Exportar a JSON
with open('/tmp/resultados_optimizacion.json', 'w') as f:
    json.dump(resultados, f, indent=2)

# Exportar a Excel
with pd.ExcelWriter('/tmp/resultados_optimizacion.xlsx', engine='openpyxl') as writer:
    # Hoja 1: Resumen
    resumen_df = pd.DataFrame([
        {'Métrica': 'Estado', 'Valor': resultados['status']},
        {'Métrica': 'Calidad Científica Total', 'Valor': f"{resultados['calidad_total']:.2f}"},
        {'Métrica': 'Equipos de Trabajo', 'Valor': resultados['recursos_utilizados']['equipos_trabajo']},
        {'Métrica': 'Horas-Hombre Totales', 'Valor': resultados['recursos_utilizados']['horas_hombre']},
        {'Métrica': 'Costo Total', 'Valor': f"${resultados['recursos_utilizados']['costo_total']:,.2f}"},
        {'Métrica': '% Presupuesto Utilizado', 'Valor': f"{resultados['recursos_utilizados']['presupuesto_utilizado_%']:.1f}%"},
    ])
    resumen_df.to_excel(writer, sheet_name='Resumen', index=False)
    
    # Hoja 2: Estrategia por círculo
    circulo_df = pd.DataFrame([
        {'Círculo': k, **v} for k, v in resultados['estrategia_por_circulo'].items()
    ])
    circulo_df.to_excel(writer, sheet_name='Por Círculo', index=False)
    
    # Hoja 3: Laboratorio
    lab_df = pd.DataFrame([
        {'Tipo de Análisis': k, 'Muestras': v} for k, v in resultados['laboratorio'].items()
    ])
    lab_df.to_excel(writer, sheet_name='Laboratorio', index=False)

print("✓ Resultados exportados:")
print("  - resultados_optimizacion.json")
print("  - resultados_optimizacion.xlsx")
print("  - resultados_optimizacion.png")